# Myeloid State Proportions per Patient and Stress-Programme Markers

Summarises the annotated myeloid object: per-patient composition of
myeloid states as a heatmap, counts per patient, and expression of
senescence, hypoxia and immunogenic cell death (ICD) markers across cell states.

In [ ]:
# !pip install --quiet scanpy
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/myeloid

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

## Config

In [ ]:
input_path = "sub_Myeloid_raw_scvi_combine_annotated.h5ad"

figures_output_dir = "./figures"
tables_output_dir = "./tables"

counts_layer = "raw_counts"
sample_col = "ID_2"
state_col = "cell_state"

# --- composition heatmap ---
heatmap_states = [
    "Monocyte_CD16high", "Macrophage INHBAhigh_infiltrating", "Macrophage INHBAhigh",
    "Macrophage INHBAhigh_inflammatory", "Macrophage CD36high", "Macrophage STAB1high", "Myeloid_dividing",
]

# "selected": % of the heatmap states only (original behaviour); "all": % of all myeloid cells in the sample
percent_denominator = "selected"
min_cells_per_sample = 20          # samples with fewer denominator cells are greyed out
sample_order = [
    "TN006_AN", "TN002_AN", "TN005_AN", "TN009_AN", "TN007_AN", "TN004_AN", "TN001_AN", "TN008_AN", "TN003_AN",
    "TN006", "TN002", "TN005", "TN009", "TN007", "TN004", "TN001", "TN008", "TN003",
    "P001", "P002", "P011", "P006", "P008", "P003", "P005", "P013", "P009", "P012",
    "E001", "O001",
    "D001", "D002", "D003",
    "R009", "R005", "R003", "R006", "R007", "R010", "R001", "R002", "R012", "R004",
    "R008", "R011",
]


# --- stress-programme dotplot ---
stress_markers = {
    "Senes": ["CDKN2A", "CDKN2B", "CDKN1A", "CDKN1B", "CDKN2D", "H2AFX", "CD33"],
    "Hypoxia": ["HIF1A", "EGLN1", "HMOX1", "SOD2", "VEGFA"],
    "ICD": ["CALR", "HMGB1"],
}

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir

## 1. Load

In [ ]:
adata = sc.read(input_path)
adata.X = adata.layers[counts_layer].copy() if counts_layer in adata.layers else adata.X.copy()
adata.uns.pop("log1p", None)

states = set(adata.obs[state_col].astype(str))
missing_states = [s for s in heatmap_states + dc_states if s not in states]
assert not missing_states, f"states not in {state_col}: {missing_states}"
missing_samples = sorted(set(adata.obs[sample_col].astype(str)) - set(sample_order))
if missing_samples:
    print("samples not in sample_order (excluded from heatmap):", missing_samples)

adata.obs[state_col].value_counts()

## 2. State composition per patient

Rows: states; columns: patients in `sample_order`. Cell value = % of the denominator
cells in that patient (see `percent_denominator`). Samples with fewer than
`min_cells_per_sample` denominator cells are shown in grey; the top bar gives n.

In [ ]:
counts = pd.crosstab(adata.obs[sample_col].astype(str), adata.obs[state_col].astype(str))
counts = counts.reindex([s for s in sample_order if s in counts.index])
selected = counts.reindex(columns=heatmap_states, fill_value=0)
denominator = selected.sum(axis=1) if percent_denominator == "selected" else counts.sum(axis=1)
percent = selected.div(denominator.replace(0, np.nan), axis=0) * 100

pd.concat([selected, denominator.rename("n_denominator"), percent.add_suffix(" (%)")], axis=1).to_csv(
    f"{tables_output_dir}/myeloid_state_percent_per_patient_{percent_denominator}.csv")

low = denominator < min_cells_per_sample
fig, axes = plt.subplots(2, 2, figsize=(12, 3.6), gridspec_kw={
    "height_ratios": [0.6, 3], "width_ratios": [40, 1], "hspace": 0.05, "wspace": 0.02})
ax_n, ax, cax = axes[0, 0], axes[1, 0], axes[1, 1]
axes[0, 1].axis("off")
ax_n.bar(np.arange(len(denominator)) + 0.5, denominator.values, color=np.where(low, "lightgrey", "black"), width=0.8)
ax_n.set_xlim(0, len(denominator))
ax_n.set_xticks([])
ax_n.set_yticks([0, int(denominator.max())])
ax_n.set_ylabel("n cells", fontsize=8)
for side in ("top", "right"):
    ax_n.spines[side].set_visible(False)

label = "% of selected states per patient" if percent_denominator == "selected" else "% of myeloid cells per patient"
sns.heatmap(percent.mask(low, axis=0).T, cmap="mako_r", linewidths=0.25, vmin=0, vmax=100,
            cbar_ax=cax, cbar_kws={"label": label}, ax=ax)
for j in np.where(low)[0]:
    ax.add_patch(plt.Rectangle((j, 0), 1, len(heatmap_states), color="lightgrey", zorder=0))
ax.set(xlabel="Patient ID", ylabel="Cell state")
ax.tick_params(axis="x", rotation=90)
plt.savefig(f"{figures_output_dir}/all_cell_type_rep_prop_by_subset_patients.pdf", bbox_inches="tight") #Figure 3
plt.show()
print(f"greyed (< {min_cells_per_sample} cells):", list(denominator.index[low]))

## 3. Senescence, hypoxia and ICD markers by cell state

Dot colour is scaled per gene across states; states are ordered by dendrogram.
Genes absent from the data are reported and skipped.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pl.umap(adata, color=state_col, size=40)

panel = {k: [g for g in v if g in adata.var_names] for k, v in stress_markers.items()}
missing = {k: sorted(set(v) - set(panel[k])) for k, v in stress_markers.items()}
if any(missing.values()):
    print("not in data:", {k: v for k, v in missing.items() if v})

axes_dict = sc.pl.dotplot(adata, {k: v for k, v in panel.items() if v}, groupby=state_col, use_raw=False,
                          standard_scale="var", cmap="Blues", figsize=(6.2, 3), dendrogram=True, show=False)
for ax in axes_dict.values():
    for tick in ax.get_xticklabels():
        tick.set_style("italic")
plt.savefig(f"{figures_output_dir}/dotplot_senescence.pdf", bbox_inches="tight") #Supplementary Figure 5
plt.show()
plt.show()